# Data cleaning

## 1. Load and inspect

This notebook will clean the data in a later section. This section only looks.

We do not fill empty cells, delete rows, or change any value here.

The file is `data/raw/online_shoppers_intention.csv`. It is the Online Shoppers Purchasing Intention Dataset from the UCI Machine Learning Repository: <https://archive.ics.uci.edu/dataset/468/online+shoppers+purchasing+intention+dataset>.

Each row is one visit to an online shop. `Revenue` is the target: did that visit end in a purchase (`True` or `False`). The other 17 columns are the inputs.

### Load the raw file

We read the file into a table and print its size. The size is the first fact we need before any cleaning choice.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

# Later steps may use random choices. This keeps those choices repeatable.
# This section does not use the seed.
np.random.seed(42)

# Print wide tables without hiding columns.
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

# The notebook may be run from the project folder or from notebooks/.
if Path("data/raw/online_shoppers_intention.csv").exists():
    raw_path = Path("data/raw/online_shoppers_intention.csv")
else:
    raw_path = Path("..") / "data" / "raw" / "online_shoppers_intention.csv"

df = pd.read_csv(raw_path)

print("File:", raw_path)
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

File: ../data/raw/online_shoppers_intention.csv
Rows: 12330
Columns: 18


The file has **12,330 rows** and **18 columns**. That is 17 input columns plus the target, `Revenue`. We did not edit the file.

### Column names and data types

A data type is the kind of value in a column: a whole number, a decimal, text, or true/false. Wrong types cause wrong summaries. For example, a mean of browser codes would be meaningless.

In [2]:
print("Column names:")
for name in df.columns:
    print("-", name)

print()
print(df.dtypes)

Column names:
- Administrative
- Administrative_Duration
- Informational
- Informational_Duration
- ProductRelated
- ProductRelated_Duration
- BounceRates
- ExitRates
- PageValues
- SpecialDay
- Month
- OperatingSystems
- Browser
- Region
- TrafficType
- VisitorType
- Weekend
- Revenue

Administrative               int64
Administrative_Duration    float64
Informational                int64
Informational_Duration     float64
ProductRelated               int64
ProductRelated_Duration    float64
BounceRates                float64
ExitRates                  float64
PageValues                 float64
SpecialDay                 float64
Month                       object
OperatingSystems             int64
Browser                      int64
Region                       int64
TrafficType                  int64
VisitorType                 object
Weekend                       bool
Revenue                       bool
dtype: object


`Month` and `VisitorType` are text. `Weekend` and `Revenue` are true/false. The rest are numbers.

`OperatingSystems`, `Browser`, `Region`, and `TrafficType` are stored as whole numbers, but each number is a code for a group. Browser `2` is not "twice" browser `1`. We leave the stored type unchanged in this section and treat them as category labels when we interpret them.

### First 5 rows

We print the first 5 visits to see real values, not just column names.

In [3]:
df.head()

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,True,False


These 5 rows are unchanged examples. Page counts and times are numbers. `Month` is a short name such as `Feb`. `VisitorType` uses an underscore, as in `Returning_Visitor`. `Weekend` and `Revenue` are `False` in these first rows.

### Missing values

A missing value is an empty cell. We count empty cells in every column before deciding whether anything needs to be filled.

In [4]:
missing = df.isna().sum()
print(missing)
print()
print("Total missing cells:", int(missing.sum()))

Administrative             0
Administrative_Duration    0
Informational              0
Informational_Duration     0
ProductRelated             0
ProductRelated_Duration    0
BounceRates                0
ExitRates                  0
PageValues                 0
SpecialDay                 0
Month                      0
OperatingSystems           0
Browser                    0
Region                     0
TrafficType                0
VisitorType                0
Weekend                    0
Revenue                    0
dtype: int64

Total missing cells: 0


Every column has **0** missing cells. The total is **0**. There is nothing to fill. This is fine.

### Duplicate rows

A duplicate row is an exact copy of another row. We only count copies here. We do not delete them in this section.

In [5]:
# Extra copies only. The first copy of each repeated row is not counted.
extra_copies = int(df.duplicated().sum())

# Every row that belongs to a repeated group, including the first copy.
rows_in_repeated_groups = int(df.duplicated(keep=False).sum())

print("Extra copied rows:", extra_copies)
print("Rows that sit in a repeated group:", rows_in_repeated_groups)

Extra copied rows: 125
Rows that sit in a repeated group: 201


There are **125** extra copied rows. **201** rows sit in a repeated group (the originals plus the copies). If we later keep one row from each group, we would remove 125 rows. We do not remove them yet.

### Numeric summary

`describe()` shows the count, average, and the smallest and largest values for each numeric column. We use it to spot negative numbers and extreme values. We do not remove extreme values here.

In [6]:
number_columns = df.select_dtypes(include="number").columns
df[number_columns].describe()

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,OperatingSystems,Browser,Region,TrafficType
count,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000
mean,2.315166,80.818611,0.503569,34.472398,31.731468,1194.746220,0.022191,0.043073,5.889258,0.061427,2.124006,2.357097,3.147364,4.069586
std,3.321784,176.779107,1.270156,140.749294,44.475503,1913.669288,0.048488,0.048597,18.568437,0.198917,0.911325,1.717277,2.401591,4.025169
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000
25%,0.000000,0.000000,0.000000,0.000000,7.000000,184.137500,0.000000,0.014286,0.000000,0.000000,2.000000,2.000000,1.000000,2.000000
50%,1.000000,7.500000,0.000000,0.000000,18.000000,598.936905,0.003112,0.025156,0.000000,0.000000,2.000000,2.000000,3.000000,2.000000
75%,4.000000,93.256250,0.000000,0.000000,38.000000,1464.157214,0.016813,0.050000,0.000000,0.000000,3.000000,2.000000,4.000000,4.000000
max,27.000000,3398.750000,24.000000,2549.375000,705.000000,63973.522230,0.200000,0.200000,361.763742,1.000000,8.000000,13.000000,9.000000,20.000000


No numeric column has a minimum below 0. The largest product-page time is about **63,974** (the unit in this file is seconds). That is a very long visit, but a long visit is possible, so we do not call it an error here.

The averages for `OperatingSystems` (about 2.12), `Browser` (about 2.36), `Region` (about 3.15), and `TrafficType` (about 4.07) are not useful. Those columns are category codes, not amounts.

### Value counts for labels

These columns name a group, even when the group is stored as a number. We count each group. `SpecialDay` is included because it only takes a few fixed values (0, 0.2, 0.4, 0.6, 0.8, and 1), not a free measurement.

In [7]:
label_columns = [
    "Month",
    "VisitorType",
    "Weekend",
    "SpecialDay",
    "OperatingSystems",
    "Browser",
    "Region",
    "TrafficType",
]

for column in label_columns:
    print("=" * 40)
    print(column)
    print(df[column].value_counts(dropna=False).sort_index())

Month
Month
Aug      433
Dec     1727
Feb      184
Jul      432
June     288
Mar     1907
May     3364
Nov     2998
Oct      549
Sep      448
Name: count, dtype: int64
VisitorType
VisitorType
New_Visitor           1694
Other                   85
Returning_Visitor    10551
Name: count, dtype: int64
Weekend
Weekend
False    9462
True     2868
Name: count, dtype: int64
SpecialDay
SpecialDay
0.0    11079
0.2      178
0.4      243
0.6      351
0.8      325
1.0      154
Name: count, dtype: int64
OperatingSystems
OperatingSystems
1    2585
2    6601
3    2555
4     478
5       6
6      19
7       7
8      79
Name: count, dtype: int64
Browser
Browser
1     2462
2     7961
3      105
4      736
5      467
6      174
7       49
8      135
9        1
10     163
11       6
12      10
13      61
Name: count, dtype: int64
Region
Region
1    4780
2    1136
3    2403
4    1182
5     318
6     805
7     761
8     434
9     511
Name: count, dtype: int64
TrafficType
TrafficType
1     2451
2     3913
3   

`Month` has **10** labels. The busiest months are May (**3,364** rows) and Nov (**2,998** rows). Feb is the smallest month in the file (**184** rows). January and April do not appear.

`VisitorType`: Returning_Visitor **10,551**, New_Visitor **1,694**, Other **85**.

`Weekend`: False **9,462**, True **2,868**.

`SpecialDay` is 0 for **11,079** rows. The other values are only 0.2, 0.4, 0.6, 0.8, and 1.0.

`OperatingSystems` uses codes 1 to 8. Code 2 is the largest group (**6,601** rows). Codes 5, 6, and 7 are rare (6, 19, and 7 rows).

`Browser` uses codes 1 to 13. Code 2 has **7,961** rows. Code 9 appears once.

`Region` uses codes 1 to 9. Code 1 has **4,780** rows.

`TrafficType` uses codes 1 to 20. Code 2 has **3,913** rows. Codes 12 and 17 appear once each.

### Target: Revenue

`Revenue` answers the project question: did the visitor buy? We need the count and the percentage of each answer. We do not rebalance the classes in this phase.

In [8]:
# True counts as 1, so sum() is the number of purchases.
bought = int(df["Revenue"].sum())
total = len(df)
did_not_buy = total - bought

print("Bought (True):", bought)
print("Did not buy (False):", did_not_buy)
print("Percent who bought: {:.2f}%".format(100 * bought / total))
print("Percent who did not buy: {:.2f}%".format(100 * did_not_buy / total))

Bought (True): 1908
Did not buy (False): 10422
Percent who bought: 15.47%
Percent who did not buy: 84.53%


**1,908** visits ended in a purchase (**15.47%**). **10,422** did not (**84.53%**). The classes are uneven. We only record that here.

### Impossible values

We look for numbers that cannot make sense: a negative count or a negative time, a time recorded for a page type the visitor did not open, and rates outside 0 to 1. Finding a problem is not the same as deleting the row.

In [9]:
print("Negative values:")
for column in number_columns:
    negative_count = int((df[column] < 0).sum())
    print(f"  {column}: {negative_count}")

page_time_pairs = [
    ("Administrative", "Administrative_Duration"),
    ("Informational", "Informational_Duration"),
    ("ProductRelated", "ProductRelated_Duration"),
]

print()
print("Page count compared with time spent:")
for pages, duration in page_time_pairs:
    time_without_pages = int(((df[pages] == 0) & (df[duration] > 0)).sum())
    pages_with_zero_time = int(((df[pages] > 0) & (df[duration] == 0)).sum())
    print(pages)
    print("  time > 0 while page count is 0:", time_without_pages)
    print("  page count > 0 while time is 0:", pages_with_zero_time)

no_pages = (
    (df["Administrative"] == 0)
    & (df["Informational"] == 0)
    & (df["ProductRelated"] == 0)
)
print()
print("Rows with all three page counts equal to 0:", int(no_pages.sum()))
print(
    "BounceRates outside 0 to 1:",
    int(((df["BounceRates"] < 0) | (df["BounceRates"] > 1)).sum()),
)
print(
    "ExitRates outside 0 to 1:",
    int(((df["ExitRates"] < 0) | (df["ExitRates"] > 1)).sum()),
)
print(
    "SpecialDay outside 0 to 1:",
    int(((df["SpecialDay"] < 0) | (df["SpecialDay"] > 1)).sum()),
)
print("PageValues below 0:", int((df["PageValues"] < 0).sum()))

Negative values:
  Administrative: 0
  Administrative_Duration: 0
  Informational: 0
  Informational_Duration: 0
  ProductRelated: 0
  ProductRelated_Duration: 0
  BounceRates: 0
  ExitRates: 0
  PageValues: 0
  SpecialDay: 0
  OperatingSystems: 0
  Browser: 0
  Region: 0
  TrafficType: 0

Page count compared with time spent:
Administrative
  time > 0 while page count is 0: 0
  page count > 0 while time is 0: 135
Informational
  time > 0 while page count is 0: 0
  page count > 0 while time is 0: 226
ProductRelated
  time > 0 while page count is 0: 0
  page count > 0 while time is 0: 717

Rows with all three page counts equal to 0: 6
BounceRates outside 0 to 1: 0
ExitRates outside 0 to 1: 0
SpecialDay outside 0 to 1: 0
PageValues below 0: 0


There are **no negative numbers**. `BounceRates`, `ExitRates`, `SpecialDay`, and `PageValues` stay inside the allowed range. That part is fine.

No row records time on a page type when that page count is 0. That check is fine.

Some rows record pages but zero time: Administrative **135**, Informational **226**, ProductRelated **717**. A viewed page should usually take some time, so these rows are odd. Zero can also mean the clock was not stored. We do not delete them in this section.

**6** rows have all three page counts equal to 0. We do not delete them here.

### Text in Month and VisitorType

The same month or visitor type can be split by a computer if the spelling, spaces, or capital letters differ. `Feb` and `feb` would be counted as two months. We print each stored label with quote marks so hidden spaces would show up.

In [10]:
print("Month labels:")
for label in sorted(df["Month"].astype(str).unique()):
    print(" ", repr(label), "rows:", int((df["Month"] == label).sum()))

month_space_rows = int((df["Month"] != df["Month"].str.strip()).sum())
print("Month rows with extra spaces:", month_space_rows)
print("Month labels if capital letters are ignored:", sorted(df["Month"].str.lower().unique()))

not_three_letters = [label for label in df["Month"].unique() if len(label) != 3]
print("Month labels that are not 3 letters:", not_three_letters)

print()
print("VisitorType labels:")
for label in sorted(df["VisitorType"].astype(str).unique()):
    print(" ", repr(label), "rows:", int((df["VisitorType"] == label).sum()))

visitor_space_rows = int((df["VisitorType"] != df["VisitorType"].str.strip()).sum())
print("VisitorType rows with extra spaces:", visitor_space_rows)
print(
    "VisitorType labels if capital letters are ignored:",
    sorted(df["VisitorType"].str.lower().unique()),
)

Month labels:
  'Aug' rows: 433
  'Dec' rows: 1727
  'Feb' rows: 184
  'Jul' rows: 432
  'June' rows: 288
  'Mar' rows: 1907
  'May' rows: 3364
  'Nov' rows: 2998
  'Oct' rows: 549
  'Sep' rows: 448
Month rows with extra spaces: 0
Month labels if capital letters are ignored: ['aug', 'dec', 'feb', 'jul', 'june', 'mar', 'may', 'nov', 'oct', 'sep']
Month labels that are not 3 letters: ['June']

VisitorType labels:
  'New_Visitor' rows: 1694
  'Other' rows: 85
  'Returning_Visitor' rows: 10551
VisitorType rows with extra spaces: 0
VisitorType labels if capital letters are ignored: ['new_visitor', 'other', 'returning_visitor']


`VisitorType` is clean. The only labels are `Returning_Visitor`, `New_Visitor`, and `Other`. There are no extra spaces and no second spelling created by capital letters.

`Month` has no extra spaces and no capital-letter copies. The format is still inconsistent: **June** is the full word (**288** rows), while the other nine months are 3-letter names such as `Jul`. January and April are absent. That is a limit of this file: it does not cover the whole year, and June is not abbreviated like the rest.

### Check the five claims

Each claim is answered TRUE or FALSE from the table above. Nothing is changed.

In [11]:
bought_percent = 100 * bought / total
# "About 15%" means within 1 percentage point of 15.
claim_1 = abs(bought_percent - 15) <= 1
print("1. About 15% of visitors bought.")
print("   Bought percent: {:.2f}%".format(bought_percent))
print("   TRUE" if claim_1 else "   FALSE")

claim_2 = extra_copies > 0
print("2. There are duplicate rows.")
print("   Extra copied rows:", extra_copies)
print("   TRUE" if claim_2 else "   FALSE")

claim_3 = df["Month"].nunique() == 10
print("3. Only 10 of 12 months appear.")
print("   Month labels:", df["Month"].nunique())
print("   Values:", sorted(df["Month"].unique().tolist()))
print("   TRUE" if claim_3 else "   FALSE")

code_columns = ["OperatingSystems", "Browser", "Region", "TrafficType"]
all_whole_numbers = all(pd.api.types.is_integer_dtype(df[column]) for column in code_columns)
print("4. OperatingSystems, Browser, Region, and TrafficType are numbers but really category labels.")
for column in code_columns:
    print(f"   {column}: type={df[column].dtype}, different codes={df[column].nunique()}")
print("   TRUE" if all_whole_numbers else "   FALSE")

claim_5 = int(df.isna().sum().sum()) == 0
print("5. There are no missing values.")
print("   Total missing cells:", int(df.isna().sum().sum()))
print("   TRUE" if claim_5 else "   FALSE")

1. About 15% of visitors bought.
   Bought percent: 15.47%
   TRUE
2. There are duplicate rows.
   Extra copied rows: 125
   TRUE
3. Only 10 of 12 months appear.
   Month labels: 10
   Values: ['Aug', 'Dec', 'Feb', 'Jul', 'June', 'Mar', 'May', 'Nov', 'Oct', 'Sep']
   TRUE
4. OperatingSystems, Browser, Region, and TrafficType are numbers but really category labels.
   OperatingSystems: type=int64, different codes=8
   Browser: type=int64, different codes=13
   Region: type=int64, different codes=9
   TrafficType: type=int64, different codes=20
   TRUE
5. There are no missing values.
   Total missing cells: 0
   TRUE


1. **TRUE.** 1,908 of 12,330 visits bought, which is 15.47%.
2. **TRUE.** There are 125 extra copied rows.
3. **TRUE.** 10 month labels appear. January and April do not.
4. **TRUE.** The four columns are whole-number codes (8, 13, 9, and 20 different codes). They name groups. We did not convert the column type in this section.
5. **TRUE.** Total missing cells: 0.

No rows were dropped or edited. The raw file is unchanged.

## 2. Cleaning

The checks above only looked. This section changes a copy called `cleaned`.
The raw file is never opened for writing.

No graph is drawn here. No scaler is fitted. The table is not split into training and test rows.


### A. Month labels

`June` is a full word. The other month labels are 3 letters. Plots later should use one style.
We change `June` to `Jun` and keep a calendar list for sorting. We do not turn the month into a number.
January and April are still absent. That limit stays.


In [12]:
# Work on a copy. df stays as the raw table loaded above.
cleaned = df.copy()
rows_start = len(cleaned)

month_order = [
    "Jan", "Feb", "Mar", "Apr", "May", "Jun",
    "Jul", "Aug", "Sep", "Oct", "Nov", "Dec",
]
print("Calendar order list:", month_order)

june_rows = int((cleaned["Month"] == "June").sum())
print('Rows labeled "June":', june_rows)

cleaned["Month"] = cleaned["Month"].replace({"June": "Jun"})

print("Month label lengths after the change:")
for label in sorted(cleaned["Month"].unique()):
    print(" ", label, len(label))

print()
print("Row count in calendar order. 0 means that month is not in the file:")
month_counts = cleaned["Month"].value_counts()
for month in month_order:
    print(f"  {month}: {int(month_counts.get(month, 0))}")


Calendar order list: ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
Rows labeled "June": 288
Month label lengths after the change:
  Aug 3
  Dec 3
  Feb 3
  Jul 3
  Jun 3
  Mar 3
  May 3
  Nov 3
  Oct 3
  Sep 3

Row count in calendar order. 0 means that month is not in the file:
  Jan: 0
  Feb: 184
  Mar: 1907
  Apr: 0
  May: 3364
  Jun: 288
  Jul: 432
  Aug: 433
  Sep: 448
  Oct: 549
  Nov: 2998
  Dec: 1727


**288** rows were labeled `June`. After the change every stored month label has 3 letters.
January and April still have **0** rows. `Month` is still text. `month_order` is only a list for later plots.


### B. Column types

`OperatingSystems`, `Browser`, `Region`, and `TrafficType` are codes for groups.
A mean of those codes does not mean anything, so we store them as text.

`Revenue` and `Weekend` become 0 and 1. `SpecialDay` stays a decimal.
It is a score from 0 to 1 for how close the visit is to a special day, not a group label.


In [13]:
code_columns = ["OperatingSystems", "Browser", "Region", "TrafficType"]

for column in code_columns:
    cleaned[column] = cleaned[column].astype(str)

cleaned["Revenue"] = cleaned["Revenue"].astype(int)
cleaned["Weekend"] = cleaned["Weekend"].astype(int)

print(cleaned.dtypes)
print()
print("Revenue values:", sorted(cleaned["Revenue"].unique().tolist()))
print("Weekend values:", sorted(cleaned["Weekend"].unique().tolist()))
print("SpecialDay min/max:", float(cleaned["SpecialDay"].min()), float(cleaned["SpecialDay"].max()))
print("Rows:", len(cleaned))


Administrative               int64
Administrative_Duration    float64
Informational                int64
Informational_Duration     float64
ProductRelated               int64
ProductRelated_Duration    float64
BounceRates                float64
ExitRates                  float64
PageValues                 float64
SpecialDay                 float64
Month                       object
OperatingSystems            object
Browser                     object
Region                      object
TrafficType                 object
VisitorType                 object
Weekend                      int64
Revenue                      int64
dtype: object

Revenue values: [0, 1]
Weekend values: [0, 1]
SpecialDay min/max: 0.0 1.0
Rows: 12330


The four code columns are text now. `Revenue` and `Weekend` are 0/1.
`SpecialDay` still runs from 0 to 1, so it stays numeric. No rows were removed.


### C. Duplicate rows

There is no ID column, so we cannot prove these 125 rows are copies.
An exact match can be one row saved twice, or two different short visits.

We still describe them before removing anything.
"Zero time and 1-2 pages" means the total recorded time is 0 and the total page count is 1 or 2.


In [14]:
extra_copy_mask = cleaned.duplicated()
extra_copies = cleaned.loc[extra_copy_mask].copy()
extra_copies["total_pages"] = (
    extra_copies["Administrative"]
    + extra_copies["Informational"]
    + extra_copies["ProductRelated"]
)
extra_copies["total_time"] = (
    extra_copies["Administrative_Duration"]
    + extra_copies["Informational_Duration"]
    + extra_copies["ProductRelated_Duration"]
)

buyers_in_copies = int(extra_copies["Revenue"].sum())
non_buyers_in_copies = int((extra_copies["Revenue"] == 0).sum())
zero_time_short_visit = int(
    ((extra_copies["total_time"] == 0) & (extra_copies["total_pages"] <= 2)).sum()
)
# Kept so the later summary can still read this name.
almost_no_activity = zero_time_short_visit

print("Exact matching rows, not proven copies:", len(extra_copies))
print("Buyers among them:", buyers_in_copies)
print("Non-buyers among them:", non_buyers_in_copies)
print("Rows with zero time and 1-2 pages:", zero_time_short_visit)
print("Page counts among those rows:")
print(extra_copies["total_pages"].value_counts().sort_index())

example_columns = [
    "Administrative",
    "Administrative_Duration",
    "Informational",
    "Informational_Duration",
    "ProductRelated",
    "ProductRelated_Duration",
    "BounceRates",
    "ExitRates",
    "PageValues",
    "Revenue",
    "total_pages",
    "total_time",
]
extra_copies[example_columns].head(5)


Exact matching rows, not proven copies: 125
Buyers among them: 0
Non-buyers among them: 125
Rows with zero time and 1-2 pages: 125
Page counts among those rows:
total_pages
1    118
2      7
Name: count, dtype: int64


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,Revenue,total_pages,total_time
158,0,0.0,0,0.0,1,0.0,0.2,0.2,0.0,0,1,0.0
159,0,0.0,0,0.0,1,0.0,0.2,0.2,0.0,0,1,0.0
178,0,0.0,0,0.0,1,0.0,0.2,0.2,0.0,0,1,0.0
418,0,0.0,0,0.0,1,0.0,0.2,0.2,0.0,0,1,0.0
456,0,0.0,0,0.0,1,0.0,0.2,0.2,0.0,0,1,0.0


There is no ID column, so these **125** rows are not proven copies.
All **125** are non-buyers (**0** buyers). All **125** have zero recorded time.
**118** have 1 page and **7** have 2 pages.

Removal is a decision under uncertainty. The reason: the same row in both a future training set and the test set would make the test score look better than a new visit would justify. None of these 125 rows is a purchase, so this decision does not drop a buyer.


In [15]:
rows_before_duplicates = len(cleaned)
buyers_before = int(cleaned["Revenue"].sum())
print("Before removal")
print("  Rows:", rows_before_duplicates)
print("  Buyers:", buyers_before)
print("  Buyer percent: {:.2f}%".format(100 * buyers_before / rows_before_duplicates))

# Stop if the copies are not the low-activity non-buyers we just counted.
if not (buyers_in_copies == 0 and almost_no_activity == len(extra_copies) and len(extra_copies) == 125):
    raise ValueError("Duplicate rows do not match the expected pattern. No rows were removed.")

cleaned = cleaned.drop_duplicates()
rows_after_duplicates = len(cleaned)
buyers_after = int(cleaned["Revenue"].sum())

print("After removal")
print("  Rows:", rows_after_duplicates)
print("  Buyers:", buyers_after)
print("  Buyer percent: {:.2f}%".format(100 * buyers_after / rows_after_duplicates))

if rows_after_duplicates != 12205:
    raise ValueError(
        f"Expected 12205 rows after removal, got {rows_after_duplicates}. Stopping."
    )


Before removal
  Rows: 12330
  Buyers: 1908
  Buyer percent: 15.47%
After removal
  Rows: 12205
  Buyers: 1908
  Buyer percent: 15.63%


Before removal: **12,330** rows, **1,908** buyers (**15.47%**).
After removal: **12,205** rows, **1,908** buyers (**15.63%**).
The buyer count did not change. The percentage rose because 125 non-buyers were removed.


### D. Consistency checks

These checks only count. They do not delete or fill values.


In [16]:
# This count is on the cleaned table, after the 125 matching rows were removed.
bounce_above_exit = int((cleaned["BounceRates"] > cleaned["ExitRates"]).sum())
print("Cleaned table rows:", len(cleaned))
print("BounceRates > ExitRates on the cleaned table:", bounce_above_exit)

zero_page_rows = cleaned.loc[
    (cleaned["Administrative"] == 0)
    & (cleaned["Informational"] == 0)
    & (cleaned["ProductRelated"] == 0)
].copy()
print("Rows with all three page counts equal to 0:", len(zero_page_rows))
zero_page_rows


Cleaned table rows: 12205
BounceRates > ExitRates on the cleaned table: 175
Rows with all three page counts equal to 0: 6


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
2683,0,0.0,0,0.0,0,0.0,0.20,0.200000,0.0,0.0,May,3,2,3,18,Returning_Visitor,1,0
3187,0,0.0,0,0.0,0,0.0,0.20,0.200000,0.0,0.0,May,2,2,8,3,Returning_Visitor,0,0
5557,0,0.0,0,0.0,0,0.0,0.20,0.200000,0.0,0.0,Nov,1,1,9,3,Returning_Visitor,0,0
5800,0,0.0,0,0.0,0,0.0,0.18,0.181818,0.0,0.0,Jun,2,2,7,6,Returning_Visitor,0,0
8052,0,0.0,0,0.0,0,0.0,0.20,0.200000,0.0,0.0,Nov,1,1,9,2,Returning_Visitor,0,0
11865,0,0.0,0,0.0,0,0.0,0.20,0.200000,0.0,0.0,Nov,1,1,8,8,Returning_Visitor,0,0


In [17]:
# A page count above 0 with a recorded time of 0.
admin_zero_time = int(((cleaned["Administrative"] > 0) & (cleaned["Administrative_Duration"] == 0)).sum())
info_zero_time = int(((cleaned["Informational"] > 0) & (cleaned["Informational_Duration"] == 0)).sum())
product_zero_time = int(((cleaned["ProductRelated"] > 0) & (cleaned["ProductRelated_Duration"] == 0)).sum())

print("Administrative pages > 0 and time = 0:", admin_zero_time)
print("Informational pages > 0 and time = 0:", info_zero_time)
print("ProductRelated pages > 0 and time = 0:", product_zero_time)


Administrative pages > 0 and time = 0: 135
Informational pages > 0 and time = 0: 226
ProductRelated pages > 0 and time = 0: 592


The **175** rows with BounceRates above ExitRates are on the cleaned table of **12,205** rows, not on the raw file. We leave them as they are.

**6** rows have all three page counts equal to 0. All 6 are non-buyers. We leave them in the file and only show them.

We keep the rows with a page count above 0 and a time of 0, because the file gives us no proof that the zero is a mistake. We do not know whether the clock failed, the time was rounded down to zero, or the page count is the value that is wrong.


### E. Missing values

There are no missing values.


In [18]:
missing_cells = int(cleaned.isna().sum().sum())
print("Missing cells:", missing_cells)


Missing cells: 0


### F. Outliers, measured only

For each numeric column we print the percent of zeros, Q1, and Q3.
Q1 is the value one quarter of the way through the column. Q3 is the value three quarters of the way through.

Where Q1 and Q3 are both 0, an IQR fence also sits at 0, so the rule only flags nonzero values. Those counts are labeled "nonzero values", not outliers. Real outliers are counted only for the other columns.

We do not delete or cap anything. A cap needs a cutoff learned from the data, so that waits until Step 5 and must be learned on training rows only.
A log shrinks large values and needs no cutoff, so it is safe, but we do not apply it here.
`Weekend` and `Revenue` are 0/1 flags, so this rule is not used on them.


In [19]:
number_columns = [
    "Administrative",
    "Administrative_Duration",
    "Informational",
    "Informational_Duration",
    "ProductRelated",
    "ProductRelated_Duration",
    "BounceRates",
    "ExitRates",
    "PageValues",
    "SpecialDay",
]

base_buyer_percent = 100 * cleaned["Revenue"].mean()
outlier_rows_for_plan = []

for column in number_columns:
    quarter_1 = float(cleaned[column].quantile(0.25))
    quarter_3 = float(cleaned[column].quantile(0.75))
    percent_zero = 100 * (cleaned[column] == 0).mean()
    spread = quarter_3 - quarter_1
    upper_fence = quarter_3 + 1.5 * spread
    lower_fence = quarter_1 - 1.5 * spread
    both_quartiles_zero = quarter_1 == 0 and quarter_3 == 0

    if both_quartiles_zero:
        # The fence is 0, so every nonzero value is flagged. That is not an outlier count.
        flagged = cleaned[column] != 0
        count_label = "nonzero values"
    else:
        flagged = (cleaned[column] < lower_fence) | (cleaned[column] > upper_fence)
        count_label = "outliers"

    flagged_count = int(flagged.sum())
    buyers_in_group = int(cleaned.loc[flagged, "Revenue"].sum())
    if flagged_count == 0:
        buyer_percent = 0.0
    else:
        buyer_percent = 100 * buyers_in_group / flagged_count

    already_zero_to_one = float(cleaned[column].max()) <= 1 and float(cleaned[column].min()) >= 0
    long_tail = (not both_quartiles_zero) and float(cleaned[column].max()) > 5 * upper_fence

    if both_quartiles_zero:
        plan = "keep"
        reason = (
            f"{percent_zero:.2f}% of rows are 0. Q1 and Q3 are both 0, so the IQR rule only flags nonzero values. "
            f"{flagged_count} nonzero rows, and {buyer_percent:.2f}% of them are buyers. "
            "These are not outliers. Keep the column. Do not cap."
        )
    elif already_zero_to_one:
        plan = "keep"
        reason = (
            f"{percent_zero:.2f}% are 0. Q1={quarter_1:.4f}, Q3={quarter_3:.4f}. "
            f"{flagged_count} real outlier rows, and {buyer_percent:.2f}% of them are buyers. "
            "The column already stays between 0 and 1. Keep it. A cap waits for Step 5."
        )
    elif long_tail:
        plan = "log transform"
        reason = (
            f"{percent_zero:.2f}% are 0. Q1={quarter_1:.4f}, Q3={quarter_3:.4f}. "
            f"{flagged_count} real outlier rows, and {buyer_percent:.2f}% of them are buyers, "
            f"against {base_buyer_percent:.2f}% of all rows. "
            "The maximum is more than 5 times the upper fence. A log would shrink that tail. "
            "Do not apply the log here. Do not cap."
        )
    else:
        plan = "keep"
        reason = (
            f"{percent_zero:.2f}% are 0. Q1={quarter_1:.4f}, Q3={quarter_3:.4f}. "
            f"{flagged_count} real outlier rows, and {buyer_percent:.2f}% of them are buyers. "
            "The maximum is not more than 5 times the upper fence. Keep the values. Do not cap."
        )

    outlier_rows_for_plan.append(
        {
            "column": column,
            "percent_zero": round(percent_zero, 2),
            "Q1": round(quarter_1, 4),
            "Q3": round(quarter_3, 4),
            "count_label": count_label,
            "count": flagged_count,
            "percent_of_rows": round(100 * flagged_count / len(cleaned), 2),
            "buyers_in_group": buyers_in_group,
            "percent_buyers": round(buyer_percent, 2),
            "plan": plan,
            "reason": reason,
        }
    )

outlier_plan = pd.DataFrame(outlier_rows_for_plan)
log_plan_columns = outlier_plan.loc[outlier_plan["plan"] == "log transform", "column"].tolist()
outlier_plan


,column,percent_zero,Q1,Q3,count_label,count,percent_of_rows,buyers_in_group,percent_buyers,plan,reason
0,Administrative,46.24,0.0000,4.0000,outliers,404,3.31,110,27.23,keep,"46.24% are 0. Q1=0.0000, Q3=4.0000. 404 real o..."
1,Administrative_Duration,47.34,0.0000,94.7000,outliers,1149,9.41,284,24.72,log transform,"47.34% are 0. Q1=0.0000, Q3=94.7000. 1149 real..."
2,Informational,78.44,0.0000,0.0000,nonzero values,2631,21.56,613,23.30,keep,"78.44% of rows are 0. Q1 and Q3 are both 0, so..."
3,Informational_Duration,80.29,0.0000,0.0000,nonzero values,2405,19.71,565,23.49,keep,"80.29% of rows are 0. Q1 and Q3 are both 0, so..."
4,ProductRelated,0.31,8.0000,38.0000,outliers,1007,8.25,289,28.70,log transform,"0.31% are 0. Q1=8.0000, Q3=38.0000. 1007 real ..."
5,ProductRelated_Duration,5.16,193.0000,1477.1548,outliers,951,7.79,289,30.39,log transform,"5.16% are 0. Q1=193.0000, Q3=1477.1548. 951 re..."
6,BounceRates,45.21,0.0000,0.0167,outliers,1428,11.70,25,1.75,keep,"45.21% are 0. Q1=0.0000, Q3=0.0167. 1428 real ..."
7,ExitRates,0.62,0.0142,0.0485,outliers,1325,10.86,8,0.60,keep,"0.62% are 0. Q1=0.0142, Q3=0.0485. 1325 real o..."
8,PageValues,77.63,0.0000,0.0000,nonzero values,2730,22.37,1538,56.34,keep,"77.63% of rows are 0. Q1 and Q3 are both 0, so..."
9,SpecialDay,89.77,0.0000,0.0000,nonzero values,1249,10.23,77,6.16,keep,"89.77% of rows are 0. Q1 and Q3 are both 0, so..."


`Informational`, `Informational_Duration`, `PageValues`, and `SpecialDay` have Q1 = 0 and Q3 = 0.
The IQR rule only flags their nonzero values, so those counts are not outliers. We keep all four.
`PageValues` has 2,730 nonzero rows, and **56.34%** of them are buyers, against **15.63%** of all rows.

Real outliers are the other six columns. `Administrative_Duration` (1,149 rows, **24.72%** buyers), `ProductRelated` (1,007 rows, **28.70%** buyers), and `ProductRelated_Duration` (951 rows, **30.39%** buyers) have a long tail, so the plan is a log later. The log is not applied here, and nothing is capped.
`Administrative` has 404 outlier rows (**27.23%** buyers) and the maximum is 27, so we keep it.
`BounceRates` has 1,428 outlier rows (**1.75%** buyers) and `ExitRates` has 1,325 (**0.60%** buyers). Both already stay between 0 and 1, so we keep them.


In this plan, **keep** means we keep the rows. We do not delete them and we do not cap them. Step 5 then applies `log1p` to every skewed column. That is a transform, not a deletion. The two agree: the rows stay, and the log only changes the shape of a long tail.


### G. Rare category codes

A rare code has fewer than 20 rows. We only list them. We do not merge codes.
The counts are from the 12,205 rows left after the extra copies were removed.


In [20]:
rare_records = []
for column in ["OperatingSystems", "Browser", "Region", "TrafficType", "VisitorType"]:
    counts = cleaned[column].value_counts()
    rare = counts[counts < 20]
    print("=" * 40)
    print(column)
    if rare.empty:
        print("  none under 20")
    else:
        print(rare)
        for label, count in rare.items():
            rare_records.append({"column": column, "code": str(label), "rows": int(count)})

print()
print("Rare codes:", len(rare_records))


OperatingSystems
OperatingSystems
6    19
7     7
5     6
Name: count, dtype: int64
Browser
Browser
12    10
11     6
9      1
Name: count, dtype: int64
Region
  none under 20
TrafficType
TrafficType
19    17
14    13
18    10
16     3
12     1
17     1
Name: count, dtype: int64
VisitorType
  none under 20

Rare codes: 12


`Region` and `VisitorType` have no code under 20 rows.
The rare codes are OperatingSystems `6` (19), `7` (7), and `5` (6); Browser `12` (10), `11` (6), and `9` (1); TrafficType `19` (17), `14` (13), `18` (10), `16` (3), `12` (1), and `17` (1).
Nothing is merged.


### Save the cleaned file

This is the full cleaned table, before any training/test split.
The file does not include the row index.


In [21]:
processed_path = raw_path.parent.parent / "processed" / "cleaned_shoppers.csv"
cleaned.to_csv(processed_path, index=False)
print("Saved:", processed_path)
print("Shape:", cleaned.shape)


Saved: ../data/processed/cleaned_shoppers.csv
Shape: (12205, 18)


### Reload check

A CSV does not remember that the four code columns are text. `read_csv` will turn codes such as `"2"` back into numbers.
Notebooks 02 and 03 must cast `OperatingSystems`, `Browser`, `Region`, and `TrafficType` back to text immediately after loading this file.
The checks below do that cast, then confirm the cleaning result.


In [22]:
check = pd.read_csv(processed_path)
code_columns = ["OperatingSystems", "Browser", "Region", "TrafficType"]
for column in code_columns:
    check[column] = check[column].astype(str)

revenue_values = sorted(int(value) for value in check["Revenue"].unique())
month_labels = set(check["Month"].unique())
expected_months = {"Feb", "Mar", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"}

assert int(check.duplicated().sum()) == 0
assert int(check.isna().sum().sum()) == 0
assert revenue_values == [0, 1]
assert len(month_labels) == 10
assert month_labels == expected_months
assert len(check) == 12205
assert len(check) == len(cleaned)
for column in code_columns:
    assert not pd.api.types.is_integer_dtype(check[column])

print("ASSERTS PASSED")
print("Reloaded shape:", check.shape)
print("Duplicate rows:", int(check.duplicated().sum()))
print("Missing cells:", int(check.isna().sum().sum()))
print("Revenue values:", revenue_values)
print("Month labels:", sorted(month_labels))
print(check[code_columns].dtypes)


ASSERTS PASSED
Reloaded shape: (12205, 18)


Duplicate rows: 0
Missing cells: 0
Revenue values: [0, 1]
Month labels: ['Aug', 'Dec', 'Feb', 'Jul', 'Jun', 'Mar', 'May', 'Nov', 'Oct', 'Sep']
OperatingSystems    object
Browser             object
Region              object
TrafficType         object
dtype: object


### Cleaning summary


In [23]:
summary = pd.DataFrame(
    [
        {
            "step": "A. Month labels",
            "what we found": f'{june_rows} rows were labeled "June". Other month labels were already 3 letters. January and April are absent.',
            "what we did": "Changed \"June\" to \"Jun\". Kept month_order, the calendar list from Jan to Dec.",
            "rows before": rows_start,
            "rows after": rows_start,
        },
        {
            "step": "B. Column types",
            "what we found": "Four code columns were whole numbers. Revenue and Weekend were true/false. SpecialDay runs from 0 to 1.",
            "what we did": "Stored the four codes as text. Stored Revenue and Weekend as 0/1. Left SpecialDay numeric.",
            "rows before": rows_start,
            "rows after": rows_start,
        },
        {
            "step": "C. Duplicate rows",
            "what we found": f"{len(extra_copies)} exact matches, not proven copies. Buyers: {buyers_in_copies}. Non-buyers: {non_buyers_in_copies}. Zero time and 1-2 pages: {almost_no_activity}.",
            "what we did": "Removed them under uncertainty, so the same row cannot sit in both train and test. No buyer was removed.",
            "rows before": rows_before_duplicates,
            "rows after": rows_after_duplicates,
        },
        {
            "step": "D. Consistency checks",
            "what we found": f"On the cleaned table, BounceRates > ExitRates: {bounce_above_exit}. All page counts 0: {len(zero_page_rows)}. Page count > 0 and time = 0: Administrative {admin_zero_time}, Informational {info_zero_time}, ProductRelated {product_zero_time}.",
            "what we did": "Counted only. Did not delete or fill any of these rows.",
            "rows before": rows_after_duplicates,
            "rows after": rows_after_duplicates,
        },
        {
            "step": "E. Missing values",
            "what we found": f"Missing cells: {missing_cells}.",
            "what we did": "Nothing. There was nothing to fill.",
            "rows before": rows_after_duplicates,
            "rows after": rows_after_duplicates,
        },
        {
            "step": "F. Outliers",
            "what we found": "Printed percent zero, Q1, and Q3. Where Q1 and Q3 are both 0, the count is nonzero values, not outliers.",
            "what we did": "Planned keep or log only. Log columns: " + ", ".join(log_plan_columns) + ". Nothing was deleted or capped.",
            "rows before": rows_after_duplicates,
            "rows after": rows_after_duplicates,
        },
        {
            "step": "G. Rare categories",
            "what we found": f"{len(rare_records)} codes have fewer than 20 rows.",
            "what we did": "Listed them. Did not merge any code.",
            "rows before": rows_after_duplicates,
            "rows after": rows_after_duplicates,
        },
    ]
)
summary


,step,what we found,what we did,rows before,rows after
0,A. Month labels,"288 rows were labeled ""June"". Other month labe...","Changed ""June"" to ""Jun"". Kept month_order, the...",12330,12330
1,B. Column types,Four code columns were whole numbers. Revenue ...,Stored the four codes as text. Stored Revenue ...,12330,12330
2,C. Duplicate rows,"125 exact matches, not proven copies. Buyers: ...","Removed them under uncertainty, so the same ro...",12330,12205
3,D. Consistency checks,"On the cleaned table, BounceRates > ExitRates:...",Counted only. Did not delete or fill any of th...,12205,12205
4,E. Missing values,Missing cells: 0.,Nothing. There was nothing to fill.,12205,12205
5,F. Outliers,"Printed percent zero, Q1, and Q3. Where Q1 and...",Planned keep or log only. Log columns: Adminis...,12205,12205
6,G. Rare categories,12 codes have fewer than 20 rows.,Listed them. Did not merge any code.,12205,12205
